In [207]:
import re
import numpy as np

PROBLEM = 'p01'
PROBLEMS_FILE = 'klotski_problems.txt'

In [208]:
class Block:
    def __init__(self, name: str, x: int, y: int, height: int, width: int):
        self.name = name
        self.x = x
        self.y = y
        self.height = height
        self.width = width

    def __str__(self):
        return f"Block {self.name} - size {self.height}x{self.width} - pos: {self.x}, {self.y}"

class Klotski:
    def __init__(self, problem_id: str):
        with open(PROBLEMS_FILE, 'r', encoding='utf8') as f_in:
            while (line := f_in.readline()).strip() != f'problem {problem_id}' and line:
                pass

            if not line:
                raise KeyError(f'Problem with id "{problem_id}" not found')

            _, n, m = f_in.readline().strip().split(' ')
            self.n, self.m = int(n), int(m)
            self.exit = np.array(f_in.readline().strip().split(' ', maxsplit=2)[-1].split(' '), dtype=int)
            self.target = f_in.readline().strip().split(' ', maxsplit=1)[-1]

            f_in.readline()
            board = list()
            for _ in range(self.n):
                board.append([x.strip() for x in re.split(r'\s+', f_in.readline().strip())])
            board = np.array(board)

            blocks = np.setdiff1d(np.unique(board), ['.'])
            self.blocks: list[Block] = list()
            for block in blocks:
                pos = np.argwhere(board == block)
                x, y = pos[0]
                height = np.ptp(pos[:, 0]) + 1
                width = np.ptp(pos[:, 1]) + 1

                self.blocks.append(Block(block, x, y, height, width))

    def get_board(self) -> np.array:
        max_len = max(len(b.name) for b in self.blocks)
        board = np.full((self.n, self.m), '.', dtype=f'<U{max_len}')
        for b in self.blocks:
            board[b.x:b.x + b.height, b.y:b.y+b.width] = b.name
        return board

    def __str__(self):
        max_len = max(len(b.name) for b in self.blocks)
        board = self.get_board()
        output = f'Size {self.n}x{self.m} - Target: {self.target}\n'
        output += '=' * ((max_len + 1) * self.m +- 1) + '\n'

        for row in board:
            output += (" ".join(s.ljust(max_len) for s in row)) + '\n'

        for i in range(self.m):
            char = ' ' if np.any(self.exit == i) else '='
            mul = max_len + (1 if i != self.m - 1 else 0)
            output += (char * mul)
        
        return output





In [209]:
def find_block_moves(problem: Klotski, board: np.array, block: Block) -> list[str]:
    moves: list[str] = list()
    x, y, h, w = block.x, block.y, block.height, block.width #aliases

    #left
    steps = 1
    can_move = True
    while can_move and y - steps >= 0:
        if np.all(board[x:x+h, y - steps] == '.'):
            moves.append(f"{block.name}-l-{steps}")
            steps += 1
        else:
            can_move = False

    #up
    steps = 1
    can_move = True
    while can_move and x - steps >= 0:
        if np.all(board[x - steps, y:y+w] == '.'):
            moves.append(f"{block.name}-u-{steps}")
            steps += 1
        else:
            can_move = False

    #right
    steps = 1
    can_move = True
    while can_move and y + steps < problem.n:
        if np.all(board[x:x+h, y + steps] == '.'):
            moves.append(f"{block.name}-r-{steps}")
            steps += 1
        else:
            can_move = False

    #down
    steps = 1
    can_move = True
    while can_move and x + steps < problem.m:
        if np.all(board[x + steps, y:y+w] == '.'):
            moves.append(f"{block.name}-d-{steps}")
            steps += 1
        else:
            can_move = False

    #exit
    if np.all(np.isin(np.arange(y, y + w), problem.exit)) and np.all(board[x + 1:, y:y+w] == '.'):
        moves.append(f"{block.name}-e-0")

    return moves

def find_moves(problem: Klotski) -> list[str]:
    moves: list[str] = list()
    board: np.array = problem.get_board()

    for block in problem.blocks:
        block_moves = find_block_moves(problem, board, block)
        moves.extend(block_moves)

    return moves


In [ ]:
problem = Klotski(PROBLEM)
print(problem)

print(find_moves(problem))

Size 3x3 - Target: A
=====
A 5 1
6 4 2
3 . .
==  =
Block 1 - size 1x1 - pos: 0, 2
Block 2 - size 1x1 - pos: 1, 2
Block 3 - size 1x1 - pos: 2, 0
Block 4 - size 1x1 - pos: 1, 1
Block 5 - size 1x1 - pos: 0, 1
Block 6 - size 1x1 - pos: 1, 0
Block A - size 1x1 - pos: 0, 0
['2-d-1', '3-r-1', '3-r-2', '4-d-1', '4-e-0']
